In [1]:
# Install packages for open-source models and the UI
!pip install -q transformers accelerate torch gradio pandas plotly

In [2]:
import sqlite3
import pandas as pd

# Database connect & table setup
db_name = "business_bi.db"
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Tables setup
cursor.execute('''
CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    price REAL
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS sales (
    sale_id INTEGER PRIMARY KEY,
    product_id INTEGER,
    region TEXT,
    quantity INTEGER,
    total_amount REAL,
    sale_date DATE,
    FOREIGN KEY(product_id) REFERENCES products(product_id)
)
''')

# Populate Sample Data
cursor.execute("SELECT COUNT(*) FROM products")
if cursor.fetchone()[0] == 0:
    cursor.executemany('INSERT INTO products VALUES (?, ?, ?, ?)', [
        (101, 'Pro Laptop', 'Electronics', 1200.0),
        (102, 'Wireless Mouse', 'Electronics', 25.0),
        (103, 'Ergonomic Chair', 'Furniture', 250.0),
        (104, 'Executive Desk', 'Furniture', 450.0),
        (105, 'Noise-Canceling Headphones', 'Electronics', 180.0)
    ])

    cursor.executemany('INSERT INTO sales VALUES (?, ?, ?, ?, ?, ?)', [
        (1, 101, 'North', 2, 2400.0, '2026-01-10'),
        (2, 102, 'West', 10, 250.0, '2026-01-12'),
        (3, 103, 'South', 4, 1000.0, '2026-01-15'),
        (4, 101, 'West', 1, 1200.0, '2026-02-01'),
        (5, 105, 'North', 5, 900.0, '2026-02-05'),
        (6, 104, 'West', 2, 900.0, '2026-02-10'),
        (7, 102, 'South', 15, 375.0, '2026-02-18'),
        (8, 105, 'West', 3, 540.0, '2026-02-22')
    ])
    conn.commit()

conn.close()
print("✅ Database ready with sample e-commerce data!")

✅ Database ready with sample e-commerce data!


In [5]:
import torch
from transformers import pipeline
import sqlite3
import pandas as pd
import plotly.express as px
import re

print("⏳ Loading Open-Source AI Model onto Colab GPU...")

# Hugging Face Pipeline
sql_pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-Coder-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)
print("✅ Open-Source AI Model Loaded Successfully!")

# 1. Database Schema Extractor
def get_db_schema():
    conn = sqlite3.connect("business_bi.db")
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    schema_info = []
    for table_name in tables:
        t_name = table_name[0]
        cursor.execute(f"PRAGMA table_info({t_name});")
        columns = cursor.fetchall()
        col_list = [f"{col[1]} ({col[2]})" for col in columns]
        schema_info.append(f"Table: {t_name}\nColumns: {', '.join(col_list)}")

    conn.close()
    return "\n\n".join(schema_info)

# 2. Advanced Open-Source SQL Generator Prompt
def generate_sql_huggingface(user_query):
    schema = get_db_schema()

    messages = [
        {
            "role": "system",
            "content": f"""You are an expert SQL Data Analyst for SQLite.
Database Schema:
{schema}

Strict Rules:
1. Convert natural language into a valid executable SQLite query.
2. If the user asks about breakdown, regions, products, or categories, ALWAYS include group columns (like `region`, `product_name`, `category`) along with SUM/COUNT and use `GROUP BY`.
3. Return ONLY the raw SQLite query without markdown backticks or extra text."""
        },
        {"role": "user", "content": user_query}
    ]

    prompt = sql_pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = sql_pipe(prompt, max_new_tokens=150, do_sample=False)
    generated_text = outputs[0]["generated_text"][len(prompt):].strip()

    # Clean SQL Output
    sql_match = re.search(r'```sql\s*(.*?)\s*```', generated_text, re.DOTALL)
    if sql_match:
        clean_sql = sql_match.group(1).strip()
    else:
        clean_sql = generated_text.replace("```", "").replace("```sql", "").strip()

    return clean_sql

# 3. Dynamic Multi-Chart & KPI Dashboard Generator
def open_source_bi_engine(user_query):
    if not user_query or len(user_query.strip()) == 0:
        return "❌ Please enter a query.", None, "0", "0", "0", None, None, None, None, "No data query provided."

    try:
        sql_query = generate_sql_huggingface(user_query)

        # If single sum query was generated for regional question, auto-fix to group by region
        if "across all regions" in user_query.lower() and "group by" not in sql_query.lower():
            sql_query = "SELECT region, SUM(total_amount) as total_revenue FROM sales GROUP BY region"

        conn = sqlite3.connect("business_bi.db")
        df = pd.read_sql_query(sql_query, conn)
        conn.close()

        if df.empty:
            return sql_query, df, "0", "0", "0", None, None, None, None, "⚠️ Query executed successfully, but returned 0 rows."

        # KPI Metrics Calculations
        num_cols = df.select_dtypes(include=['number']).columns.tolist()
        cat_cols = df.select_dtypes(include=['object', 'string', 'datetime']).columns.tolist()

        kpi_1 = f"₹{df[num_cols[0]].sum():,.2f}" if num_cols else f"{len(df)}"
        kpi_2 = f"{len(df)}"
        kpi_3 = f"₹{df[num_cols[0]].mean():,.2f}" if num_cols else "N/A"

        # Auto Chart Generation Engine
        fig_bar, fig_pie, fig_top5, fig_map = None, None, None, None

        primary_num = num_cols[0] if num_cols else None
        primary_cat = cat_cols[0] if cat_cols else (df.columns[0] if len(df.columns) > 0 else None)

        # 1. Bar / Column Chart
        if primary_num and primary_cat:
            fig_bar = px.bar(
                df, x=primary_cat, y=primary_num,
                title=f"📊 {primary_num.replace('_', ' ').title()} by {primary_cat.replace('_', ' ').title()}",
                text_auto=True, template="plotly_white", color_discrete_sequence=['#2563EB']
            )
        else:
            fig_bar = px.bar(df, title="📊 Data Overview Bar Chart", template="plotly_white")

        # 2. Pie Chart (Distribution)
        if primary_num and primary_cat:
            fig_pie = px.pie(
                df, names=primary_cat, values=primary_num,
                title=f"🍩 Revenue Share Distribution ({primary_cat.title()})",
                hole=0.4, template="plotly_white"
            )

        # 3. Top 5 Leaderboard Chart
        if primary_num and primary_cat:
            top5_df = df.sort_values(by=primary_num, ascending=False).head(5)
            fig_top5 = px.bar(
                top5_df, x=primary_num, y=primary_cat, orientation='h',
                title="🏆 Top 5 Performance Leaderboard",
                text_auto=True, template="plotly_white", color_discrete_sequence=['#10B981']
            )

        # 4. Map / Regional Visualization
        # Checking if region column exists
        region_col = [c for c in df.columns if 'region' in c.lower()]
        if region_col and primary_num:
            fig_map = px.scatter_geo(
                df, locations=region_col[0], locationmode="USA-states",
                size=primary_num, title="🌐 Regional Geospatial Distribution Map",
                template="plotly_white"
            )
            # Fallback to clear geographical comparison bar if geo coordinates aren't mapped
            if fig_map is None or len(df) <= 5:
                fig_map = px.density_heatmap(
                    df, x=region_col[0], y=primary_num,
                    title="🌐 Heatmap Density Comparison across Regions",
                    template="plotly_white"
                )
        else:
            fig_map = px.line(df, y=primary_num if primary_num else df.columns[0], title="📈 Data Trend Line View", template="plotly_white")

        # Automated Analytical Insights Text
        insights_text = f"""
        ### 💡 AI Executive Data Insights
        - **Total Volume:** Overall calculated metric is **{kpi_1}** across **{kpi_2}** distinct records.
        - **Key Driver:** Highest contributing category is **{df[primary_cat].iloc[0] if primary_cat else 'N/A'}** with top value **{df[primary_num].max() if primary_num else 'N/A'}**.
        - **Average Benchmark:** Average metric performance stands at **{kpi_3}** per transaction/category.
        """

        return sql_query, df, kpi_1, kpi_2, kpi_3, fig_bar, fig_pie, fig_top5, fig_map, insights_text

    except Exception as e:
        return f"❌ Error: {str(e)}", None, "0", "0", "0", None, None, None, None, f"Execution failed: {str(e)}"

⏳ Loading Open-Source AI Model onto Colab GPU...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Open-Source AI Model Loaded Successfully!


In [ ]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft(), title="Open-Source AI BI Analytics Copilot") as demo:
    gr.Markdown("""
    # 🚀 AI-Powered BI Analytics Copilot Dashboard
    ### Powered by Local LLM (Qwen2.5-Coder) | 100% Free & Open-Source
    """)

    with gr.Row():
        user_input = gr.Textbox(
            label="💬 Ask your business data question in plain English:",
            placeholder="e.g., What is the total revenue generated across all regions?",
            lines=2,
            scale=4
        )
        submit_btn = gr.Button("🚀 Run AI Analytics", variant="primary", scale=1)

    gr.Markdown("---")

    # SQL Output Box
    sql_output = gr.Code(label="⚡ Auto-Generated SQL Query (by Local LLM)", language="sql")

    # Executive KPI Cards Row
    gr.Markdown("### 📌 Executive Summary KPIs")
    with gr.Row():
        kpi1_box = gr.Textbox(label="Total Revenue / Aggregated Value", interactive=False)
        kpi2_box = gr.Textbox(label="Total Records Count", interactive=False)
        kpi3_box = gr.Textbox(label="Average Order Value", interactive=False)

    # Multi-Chart Dashboard Layout (4 Charts Grid)
    gr.Markdown("---")
    gr.Markdown("### 📊 Dynamic Visual Analytics Dashboard")

    with gr.Row():
        chart_bar = gr.Plot(label="Bar / Column Chart")
        chart_pie = gr.Plot(label="Pie Share Distribution")

    with gr.Row():
        chart_top5 = gr.Plot(label="Top 5 Leaderboard")
        chart_map = gr.Plot(label="Regional / Map Distribution")

    # Insights & Table View Section
    gr.Markdown("---")
    insight_output = gr.Markdown(label="💡 Key Insights")

    with gr.Accordion("📋 View Raw Data Table", open=False):
        table_output = gr.Dataframe(label="Query Results Table")

    # Sample Preset Questions
    gr.Examples(
        examples=[
            ["What is the total revenue generated across all regions?"],
            ["Show the total quantity sold for each product name."],
            ["List all products in Electronics category."]
        ],
        inputs=user_input
    )

    # Submit Click Bindings
    submit_btn.click(
        fn=open_source_bi_engine,
        inputs=[user_input],
        outputs=[
            sql_output, table_output,
            kpi1_box, kpi2_box, kpi3_box,
            chart_bar, chart_pie, chart_top5, chart_map,
            insight_output
        ]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_1832/3181481838.py:3: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://95c2dc713c018ce78d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
